# Artefactual Package Demo: Hallucination Detection with EPR 

This notebook demonstrates the `artefactual` package for scoring LLM outputs, specifically focusing on hallucination detection using entropy-based methods. Here we will use EPR (Entropy Production Rate), which computes entropy at each token and averages it across the entire sequence.

We explore two examples:
1.  **General Knowledge Question**: 

    *"What is the capital city of France?"* → `"Paris."` (expected high certainty, low entropy expected)

2.  **Hallucination Trigger**: Asking about the first author of our paper (Charles Moslonka) to observe how the model hallucinates biographical details.

    *"Who is Charles Moslonka?"* → a fabricated biography (expected high uncertainty, high entropy expected)

We will use:
* **JSON fixture (open_ai_responses.json):** two mock OpenAI Responses API outputs with top logprobs per token.
* **Local Weight Fixture (calibration_ministral.json):** pre-trained coefficients for the WEPR scorer
* **EPR :** scorer from the artefactual package via the scikit-learn pipeline API


In [34]:
import json
from pathlib import Path

from artefactual.scoring.base_detector import epr

In [ ]:
WEIGHTS_PATH = "calibration_ministral.json"
DATA_PATH = "open_ai_responses.json"

## Load Example Responses

The fixture contains two responses to illustrate the contrast between a certain and a uncertain answer.

In [ ]:
fixture_path = Path(DATA_PATH)
with fixture_path.open(encoding="utf-8") as f:
    data = json.load(f)

responses = data["responses"]
print(f"Loaded {len(responses)} responses")

## Build the EPR Pipeline

Calibration weights are loaded from the package. `epr()` always requires a `pretrained_model_name_or_path`.

In [ ]:
detector = epr(pretrained_model_name_or_path=WEIGHTS_PATH)
detector

## Sequence-Level Scoring

`predict_proba(response)` returns an array of shape `(n_sequences, 2)`.
Column 1 is the hallucination probability, higher means the model was more uncertain.

In [37]:
for resp in responses:
    prompt = resp.get("metadata", {}).get("prompt", "")
    text = resp["output"][0]["content"][0]["text"]
    score = detector.predict_proba(resp)[:, 1][0]
    print(f"Prompt : {prompt}")
    print(f"Answer : {text}")
    print(f"EPR score: {score:.2f}")
    print()

Prompt : What is the capital city of France? Please answer briefly.
Answer : Paris.
EPR score: 0.22

Prompt : Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.
Answer : Charles Moslonka is a French singer born in Lyon in 1985.
EPR score: 1.00



## Token Level Scoring

`predict_token_proba(response)` returns an array of shape `(n_sequences, max_tokens, 1)`.

* **`n_sequences`**: Response index.
* **`max_tokens`**: Token index within the sequence.
* **`1`**: The per-token scalar hallucination probability.


In [38]:
for resp in responses:
    prompt = resp.get("metadata", {}).get("prompt", "")
    token_scores = detector.predict_token_proba(resp)  # shape: (1, max_tokens, 1)
    scores = token_scores[0, :, 0]

    # Extract token strings from the response logprobs
    tokens = [t["token"] for t in resp["output"][0]["content"][0]["logprobs"]]

    print(f"Prompt: {prompt}")
    for token, score in zip(tokens, scores):
        print(f"  {token!r:13s} → {score:.2f}")
    print()

Prompt: What is the capital city of France? Please answer briefly.
  'Paris'       → 0.10
  '.'           → 0.53
  '</s>'        → 0.17

Prompt: Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.
  'Charles'     → 1.00
  ' Moslonka'   → 1.00
  ' is'         → 1.00
  ' a'          → 1.00
  ' French'     → 1.00
  ' singer'     → 1.00
  ' born'       → 1.00
  ' in'         → 1.00
  ' Lyon'       → 1.00
  ' in 1985'    → 1.00
  '.'           → 1.00

